In [1]:
# Day 64 — Table to Natural Language Conversion
## Fixing RAG retrieval for financial tables

In [10]:
from bs4 import BeautifulSoup
import re

In [23]:
class TableToNaturalLanguage:
    def __init__(self,company,fiscal_years):
        """
        Initializes the converter with specific document metadata.
        """
        self.company = company
        self.fiscal_years = fiscal_years

    def is_financial_table(self,table_soup):
        """
        Analyzes a table to determine if it contains financial data 
        based on keyword presence and numerical patterns.
        """
        financial_keywords = ["net", "revenue", "income", "sales", "earnings", 
                          "profit", "loss", "assets", "cash", "cost"]
        numeric_pattern =re.compile(r'\$?[\d,]+\.?\d*')

        rows = table_soup.find_all("tr")

        for row in rows:
            cells = row.find_all("td")
            if len(cells) < 2:
                continue

            first_cell_text = cells[0].get_text(strip=True).lower()
            has_keyword = any(kw in first_cell_text for kw in financial_keywords)

            has_numeric_value = False
            for cell in cells[1:]:
                cell_text = cell.get_text(strip=True)
                # We look for a cell that isn't empty and matches our financial number pattern
                if cell_text and numeric_pattern.search(cell_text):
                    has_numeric_value = True
                    break

            # If a single row contains both a keyword and a financial number, it's a winner
            if has_keyword and has_numeric_value:
                return True

        return False

    def convert(self, raw_html):
        """
        Parses HTML table rows and converts them into natural language f-strings.
        Returns: list of strings.
        """
        soup = BeautifulSoup(raw_html,"html.parser")
        if not self.is_financial_table(soup):
            return []
        rows = soup.find_all("tr")

        results = []

        for row in rows:
            cells = row.find_all("td")

            if len(cells)<2:
                continue

            metric = cells[0].get_text(strip=True)

            if not metric:
                continue

            for i ,  year in  enumerate(self.fiscal_years):
                if i+1 < len(cells):
                    value = cells[i+1].get_text(strip=True)

                    sentence = f"The {metric} of {self.company} in the fiscal year {year} was {value}."
                    results.append(sentence)
        return results

In [24]:
test_html = """
<table>
  <tr>
    <td>Net sales</td>
    <td>383,285</td>
    <td>394,328</td>
  </tr>
  <tr>
    <td>Net income</td>
    <td>96,995</td>
    <td>99,803</td>
  </tr>
</table>
"""

In [25]:
converter = TableToNaturalLanguage(company="Apple", fiscal_years=[2023, 2022])

In [26]:
prose_results = converter.convert(test_html)

In [27]:
print("--- Natural Language Conversion Results ---")
for i, sentence in enumerate(prose_results, 1):
    print(f"{i}. {sentence}")

--- Natural Language Conversion Results ---
1. The Net sales of Apple in the fiscal year 2023 was 383,285.
2. The Net sales of Apple in the fiscal year 2022 was 394,328.
3. The Net income of Apple in the fiscal year 2023 was 96,995.
4. The Net income of Apple in the fiscal year 2022 was 99,803.


In [28]:
soup = BeautifulSoup(open("apple_10k_2023.htm", encoding="utf-8", errors="ignore").read(), "html.parser")

tables = soup.find_all("table")
for table in tables:
    if converter.is_financial_table(table):
        print(table.prettify()[:2000])
        break

<table style="border-collapse:collapse;display:inline-table;margin-bottom:5pt;vertical-align:text-bottom;width:100.000%">
 <tr>
  <td style="width:1.0%">
  </td>
  <td style="width:41.297%">
  </td>
  <td style="width:0.1%">
  </td>
  <td style="width:1.0%">
  </td>
  <td style="width:10.595%">
  </td>
  <td style="width:0.1%">
  </td>
  <td style="width:0.1%">
  </td>
  <td style="width:0.530%">
  </td>
  <td style="width:0.1%">
  </td>
  <td style="width:1.0%">
  </td>
  <td style="width:8.695%">
  </td>
  <td style="width:0.1%">
  </td>
  <td style="width:0.1%">
  </td>
  <td style="width:0.530%">
  </td>
  <td style="width:0.1%">
  </td>
  <td style="width:1.0%">
  </td>
  <td style="width:10.595%">
  </td>
  <td style="width:0.1%">
  </td>
  <td style="width:0.1%">
  </td>
  <td style="width:0.530%">
  </td>
  <td style="width:0.1%">
  </td>
  <td style="width:1.0%">
  </td>
  <td style="width:8.695%">
  </td>
  <td style="width:0.1%">
  </td>
  <td style="width:0.1%">
  </td>
  <